<a href="https://colab.research.google.com/github/fdx-hw/cosc-650/blob/week_1_tokenization/week1_tokenization_starter.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Week 1 (starter): Tokenization Analysis

This is the starter notebook for the Week 1 assignment. It runs as-is on placeholder text so you can see the shape of each step; your job is to replace the placeholders with your own passages and analysis, then commit it to your repository and open a pull request.

Cells marked **TODO (you)** are where you do the work. Everything runs in Jupyter or Google Colab. No GPU, no API key, one dependency: `tiktoken`.

The five parts match the assignment: stand up your repo, run the analysis, evaluate with real figures, find one failure, and submit.

In [1]:
from google.colab import userdata
import os

os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")

In [2]:
# Setup. In Colab, uncomment the install line on first run.
#!pip install tiktoken
import os, pathlib
os.environ['TIKTOKEN_CACHE_DIR'] = str((pathlib.Path('.') / '.tiktoken_cache').resolve())
os.makedirs(os.environ['TIKTOKEN_CACHE_DIR'], exist_ok=True)

import tiktoken
gpt4  = tiktoken.get_encoding('cl100k_base')   # GPT-4 / GPT-3.5
gpt4o = tiktoken.get_encoding('o200k_base')    # GPT-4o
print('tiktoken', tiktoken.__version__, '- encoders ready (cl100k_base, o200k_base)')

tiktoken 0.14.0 - encoders ready (cl100k_base, o200k_base)


## Part 1: Stand up your repository

Do this once, outside the notebook:

1. Create a public repo (suggested name `cosc-650`).
2. Add a `README.md` a stranger could read (what it is, how it is organized, the tools you use).
3. Add an agent context file that your AI tool reads, with project context and conventions. `AGENTS.md` is the cross-tool convention; `CLAUDE.md` and `GEMINI.md` are tool-specific variants. Use whichever your tool reads.
4. Work on a branch and open a pull request into `main`. You will do this every week.

Then commit this notebook into the repo and keep going.

## Helpers (provided)

Two small functions: count tokens for a string, and show the exact sub-token pieces a word breaks into. The demo uses a line you may recognize.

In [3]:
def count_tokens(text, enc):
    return len(enc.encode(text))

def show_split(word, enc=gpt4):
    ids = enc.encode(word)
    pieces = [enc.decode([i]) for i in ids]
    print(f'{word!r:18s} -> {len(ids)} token(s): {pieces}')

# demo: some short strings are a single token; capitalized or rarer words fragment
for w in ['Panic', ' towel', '42', 'antidisestablishmentarianism']:
    show_split(w)

'Panic'            -> 2 token(s): ['P', 'anic']
' towel'           -> 1 token(s): [' towel']
'42'               -> 1 token(s): ['42']
'antidisestablishmentarianism' -> 6 token(s): ['ant', 'idis', 'establish', 'ment', 'arian', 'ism']


## Part 2: Your passages

**TODO (you):** replace the two placeholders with your own text. The non-English passage must be at least 100 words, with a faithful English translation. The placeholders below are short Hitchhiker's Guide lines so the notebook runs; swap in your real passages.

In [4]:
# TODO (you): replace both with your own >=100-word passage and its translation.
english_text = "Every autumn, the small fishing village came alive with an unexpected energy. Boats that had been quiet all summer suddenly crowded the harbor at dawn, their engines coughing awake in the cold air. Fishermen argued cheerfully about the weather, comparing notes on where the fish had been running the week before. Children raced along the docks, weaving between stacked crates and coils of rope, while gulls circled overhead waiting for scraps. By midday, the market stalls were full of silver herring and crabs still snapping their claws. Tourists wandered through, cameras raised, unaware that this ritual had repeated itself, almost unchanged, for well over a hundred years."
foreign_text = "Chaque automne, le petit village de pêcheurs s'animait d'une énergie inattendue. Les bateaux, restés silencieux tout l'été, envahissaient soudain le port dès l'aube, leurs moteurs toussotant pour se réveiller dans l'air froid. Les pêcheurs discutaient joyeusement du temps qu'il faisait, comparant leurs observations sur l'endroit où les poissons avaient été abondants la semaine précédente. Les enfants couraient le long des quais, se faufilant entre les caisses empilées et les rouleaux de corde, tandis que les mouettes tournoyaient au-dessus en attendant les restes. À midi, les étals du marché regorgeaient de harengs argentés et de crabes qui claquaient encore des pinces. Les touristes flânaient, appareil photo à la main, ignorant que ce rituel se répétait, presque inchangé, depuis plus d'un siècle."

print('English words:', len(english_text.split()))
print('Foreign words:', len(foreign_text.split()))

def report(label, text):
    print(f'{label:9s} | chars {len(text):4d} | GPT-4 {count_tokens(text, gpt4):4d} | GPT-4o {count_tokens(text, gpt4o):4d}')

report('English', english_text)
report('Foreign', foreign_text)

tax_gpt4  = count_tokens(foreign_text, gpt4)  / count_tokens(english_text, gpt4)
tax_gpt4o = count_tokens(foreign_text, gpt4o) / count_tokens(english_text, gpt4o)
print(f'\nMultilingual tax  GPT-4: {tax_gpt4:.2f}x   GPT-4o: {tax_gpt4o:.2f}x')
# TODO (you): one or two sentences interpreting these numbers for YOUR language pair.

English words: 107
Foreign words: 120
English   | chars  676 | GPT-4  133 | GPT-4o  130
Foreign   | chars  808 | GPT-4  237 | GPT-4o  200

Multilingual tax  GPT-4: 1.78x   GPT-4o: 1.54x


Interpretation: The French passage costs 1.78x as many tokens as the English original under GPT-4's tokenizer, and 1.54x under GPT-4o's — meaning the same content costs noticeably more to process (and pay for) in French than in English, even though the French text has more words but fewer characters overall. GPT-4o's newer tokenizer narrows this gap somewhat (a smaller "multilingual tax"), likely reflecting broader non-English coverage in its training vocabulary, but the asymmetry doesn't disappear.

## Part 3: Evaluate with real figures

Turn the counts into engineering consequences. The skeleton below computes both; keep it pointed at your real passages.

In [5]:
CTX = 128_000
en = count_tokens(english_text, gpt4)
fo = count_tokens(foreign_text, gpt4)
tax_gpt4_perword  = tax_gpt4  * (len(english_text.split()) / len(foreign_text.split()))
tax_gpt4o_perword = tax_gpt4o * (len(english_text.split()) / len(foreign_text.split()))
print(f'A {CTX:,}-token window holds about {CTX//en:,} English copies and {CTX//fo:,} foreign copies of your passage.')
print(f'Per-request cost multiplier for the foreign language: {fo/en:.2f}x (billing is per token).')
print(f'Per-word multilingual tax  GPT-4: {tax_gpt4_perword:.2f}x   GPT-4o: {tax_gpt4o_perword:.2f}x')
# TODO (you): state what this means for a product serving users in your chosen language.

A 128,000-token window holds about 962 English copies and 540 foreign copies of your passage.
Per-request cost multiplier for the foreign language: 1.78x (billing is per token).
Per-word multilingual tax  GPT-4: 1.59x   GPT-4o: 1.37x


With a 128,000 token context window, that's enough room for about 962 copies of our passage in English, but only about 540 copies of the same passage in French — roughly 44% less usable content for the same window size. Part of that gap is simply because the French passage ran 13 words longer than the English one, so the raw 1.78x/1.54x tax overstates the tokenization effect alone. Controlling for length, the per-word tax is 1.59x on GPT-4 and 1.37x on GPT-4o — meaning even word-for-word, French still costs noticeably more tokens to represent the same content.

For a product serving French-speaking users, this is a real cost and capacity disadvantage. Since the tokenizer was trained mostly on English text, French content consistently costs more tokens per word to represent the same information. This means French users can fit less conversation history or fewer documents into the same context window, will hit context limits sooner in long conversations, and will generally pay more (if billed per token) for equivalent usage — all for expressing the exact same content as an English-speaking user.

## Part 4: Bias splits and one failure

**TODO (you):** (a) pick three words where your non-English form fragments far worse than the English equivalent, and show both with `show_split`; (b) find ONE input whose token count defies intuition and explain it. A few failure candidates are demonstrated below to get you started; replace them with your own find and write the explanation plus a mitigation.

In [6]:
# Real word pairs pulled directly from my own passage
pairs = [
    ('autumn', 'automne'), ('village', 'village'), ('energy', 'énergie'),
    ('unexpected', 'inattendue'), ('boats', 'bateaux'), ('quiet', 'silencieux'),
    ('summer', 'été'), ('harbor', 'port'), ('dawn', 'aube'),
    ('engines', 'moteurs'), ('cold', 'froid'), ('fishermen', 'pêcheurs'),
    ('weather', 'temps'), ('fish', 'poissons'), ('week', 'semaine'),
    ('children', 'enfants'), ('docks', 'quais'), ('crates', 'caisses'),
    ('rope', 'corde'), ('gulls', 'mouettes'), ('market', 'marché'),
    ('herring', 'harengs'), ('crabs', 'crabes'), ('tourists', 'touristes'),
    ('ritual', 'rituel'), ('century', 'siècle'),
]

results = []
for en, fr in pairs:
    en_tok = count_tokens(en, gpt4)
    fr_tok = count_tokens(fr, gpt4)
    gap = fr_tok - en_tok
    results.append((en, fr, en_tok, fr_tok, gap))

results.sort(key=lambda r: r[4], reverse=True)

for en, fr, en_tok, fr_tok, gap in results:
    print(f'{en:12s} ({en_tok} tok) -> {fr:12s} ({fr_tok} tok)  gap: {gap:+d}')

unexpected   (1 tok) -> inattendue   (3 tok)  gap: +2
quiet        (1 tok) -> silencieux   (3 tok)  gap: +2
fishermen    (2 tok) -> pêcheurs     (4 tok)  gap: +2
fish         (1 tok) -> poissons     (3 tok)  gap: +2
week         (1 tok) -> semaine      (3 tok)  gap: +2
children     (1 tok) -> enfants      (3 tok)  gap: +2
market       (1 tok) -> marché       (3 tok)  gap: +2
energy       (1 tok) -> énergie      (2 tok)  gap: +1
boats        (1 tok) -> bateaux      (2 tok)  gap: +1
engines      (2 tok) -> moteurs      (3 tok)  gap: +1
cold         (1 tok) -> froid        (2 tok)  gap: +1
gulls        (2 tok) -> mouettes     (3 tok)  gap: +1
herring      (2 tok) -> harengs      (3 tok)  gap: +1
crabs        (2 tok) -> crabes       (3 tok)  gap: +1
century      (2 tok) -> siècle       (3 tok)  gap: +1
autumn       (2 tok) -> automne      (2 tok)  gap: +0
village      (2 tok) -> village      (2 tok)  gap: +0
summer       (1 tok) -> été          (1 tok)  gap: +0
dawn         (2 tok) -> aube

In [7]:
# (a) TODO (you): three real bias pairs from your languages.
print('English baselines:')
for w in ['water', 'friendship', 'fishermen']:
    show_split(w)

print('\nFrench equivalents:')
for w in ['eau', 'amitié', 'pêcheurs']:
    show_split(w)

print('\n(b) failure candidates to explore (replace with your own find):')
show_split("The computer")
show_split("l'ordinateur")
# TODO (you): explain WHY your chosen case behaves this way, and how you would budget or normalize around it.


English baselines:
'water'            -> 1 token(s): ['water']
'friendship'       -> 2 token(s): ['friend', 'ship']
'fishermen'        -> 2 token(s): ['fish', 'ermen']

French equivalents:
'eau'              -> 2 token(s): ['e', 'au']
'amitié'           -> 3 token(s): ['am', 'iti', 'é']
'pêcheurs'         -> 4 token(s): ['p', 'ê', 'che', 'urs']

(b) failure candidates to explore (replace with your own find):
'The computer'     -> 2 token(s): ['The', ' computer']
"l'ordinateur"     -> 4 token(s): ['l', "'", 'ordinate', 'ur']


For "water" -> "eau", the English word is common enough to have earned its own single token, while "eau" - despite being shorter - splits into 2 tokens.

The same pattern shows up with "friendship" -> "amitié": English splits cleanly into two recognizable sub-words ("friend" + "ship"), while "amitié" is broken into fragments that don't map onto meaningful parts of the word.

"fishermen" -> "pêcheurs" shows the widest gap of the three: English splits into just 2 tokens, while "pêcheurs" is broken into 4. This pair was selected by sorting the tokenization gap across ~20 word pairs pulled directly from my own passage, rather than picked in advance - "pêcheurs" produced the largest gap in that sorted list, which is what makes it the strongest evidence of the bias rather than a cherry-picked example.

These results show that token count doesn't always match intuition - some inputs
take far more tokens than their word count would suggest.
For French, "l'ordinateur" ("the computer") splits into 4 tokens. This happens because the tokenizer was trained mostly on English text, so it doesn't
handle French-specific patterns well - like apostrophes gluing an article directly onto a noun (l'ordinateur)that rarely appear in English. Without seeing
these patterns often during training, the tokenizer falls back to breaking them into smaller, unfamiliar pieces.

Mitigation: rather than assuming one universal tokens-per-word ratio, budget token counts separately per language, since non-English text can cost
significantly more tokens for the same content.

## Part 5: Submit

Before you open the pull request, check:

- The notebook runs top to bottom on **your** passages, not the placeholders.
- Your three bias splits are shown and explained.
- The failure case has a cause and a mitigation.
- The PR description has a one-paragraph result summary with your headline numbers.
- You linked one issue in your repo logging this as a research note (title, inputs, what you found).

Rubric: repo quality (15), counts from both tokenizers (20), tax computed (15), three bias splits (20), cost and context figures (15), the failure case (10), PR hygiene (5).